# Understanding PPO/RLHF - Deep Dive

This notebook demonstrates how PPO (Proximal Policy Optimization) works for RLHF (Reinforcement Learning from Human Feedback).

## What is PPO?

PPO is a reinforcement learning algorithm that optimizes a language model to maximize reward while staying close to a reference policy.

**Key Differences from DPO:**
- **DPO**: Single-phase supervised learning (simple, stable)
- **PPO**: Two-phase RL algorithm (complex, powerful)

## The RLHF Pipeline

```
Stage 1: SFT → Stage 2: Reward Model → Stage 3: PPO (this notebook)
```

## Key Concepts

1. **Four Models**: Actor, Critic, Reference, Reward Model
2. **Two Phases**: Rollout (generate data) + Update (optimize policy)
3. **Clipped Objective**: Prevents catastrophic policy changes
4. **GAE**: Computes advantages with bias-variance trade-off
5. **KL Penalty**: Keeps policy close to reference

Let's see it in action!

## Part 1: Setup

In [ ]:
# Setup
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.models.language import LanguageModel
from src.models.reward import RewardModel
from src.core.ppo import (
    ppo_loss,
    value_loss,
    compute_gae,
    RolloutBuffer,
    PPOConfig,
    PPOTrainer,
    compute_log_probs,
    compute_rlhf_reward,
)
from src.data.processors.preference import create_synthetic_preference_data

print("✅ Imports successful")

## Part 2: The Four Models

PPO requires managing four models simultaneously:

1. **Actor (Policy)**: Model being optimized (trainable)
2. **Critic (Value Function)**: Estimates expected rewards (trainable)
3. **Reference**: Frozen copy of initial policy (for KL penalty)
4. **Reward Model**: Scores responses (pre-trained from Phase 3)

### Why Four Models?

- **Actor**: Generates responses, updated via PPO
- **Critic**: Predicts future rewards, helps compute advantages
- **Reference**: Provides baseline, prevents drift
- **Reward Model**: Provides training signal

Let's load them all!

In [ ]:
print("Loading the four models...\n")

# 1. Actor (policy) - trainable
print("[1/4] Actor (Policy Model)...")
actor = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=True,
    lora_config={
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "target_modules": ["c_attn", "c_proj"],
        "bias": "none",
        "task_type": "CAUSAL_LM",
    },
)
print(f"  ✅ Actor: {actor.num_trainable_parameters:,} trainable params\n")

# 2. Critic (value function) - trainable
print("[2/4] Critic (Value Function)...")
critic_base = LanguageModel.from_pretrained(
    "gpt2",
    use_lora=True,
    lora_config={
        "r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "target_modules": ["c_attn", "c_proj"],
        "bias": "none",
        "task_type": "CAUSAL_LM",
    },
)
critic = RewardModel(base_model=critic_base, freeze_base=False)
print(f"  ✅ Critic: {critic.num_trainable_parameters:,} trainable params\n")

# 3. Reference - frozen
print("[3/4] Reference Model (frozen)...")
reference = LanguageModel.from_pretrained("gpt2", use_lora=False)
for p in reference.model.parameters():
    p.requires_grad = False
print(f"  ✅ Reference: frozen (baseline policy)\n")

# 4. Reward Model - frozen
print("[4/4] Reward Model (frozen)...")
rm_base = LanguageModel.from_pretrained("gpt2", use_lora=False)
reward_model = RewardModel(base_model=rm_base, freeze_base=True)
print(f"  ✅ Reward Model: frozen (trained in Phase 3)\n")

print("\n✅ All four models loaded successfully!")
print(f"\nMemory check: {actor.num_parameters + critic.num_parameters:,} trainable params total")

## Part 3: Understanding PPO Loss (The Clipped Objective)

The heart of PPO is the **clipped surrogate objective**:

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min(r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t) \right]$$

Where:
- $r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{old}}(a_t | s_t)}$ (probability ratio)
- $A_t$ = advantage (how much better than expected)
- $\epsilon$ = clip range (typically 0.2)

### Why Clipping?

**Without clipping**: Large policy updates → mode collapse
**With clipping**: Bounded updates → stable learning

Let's visualize!

In [ ]:
# Visualize PPO clipping
def plot_ppo_clipping(clip_range=0.2):
    ratios = np.linspace(0.5, 2.0, 100)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Positive advantage (good action)
    advantage_pos = 1.0
    loss_unclipped_pos = ratios * advantage_pos
    loss_clipped_pos = np.minimum(
        loss_unclipped_pos,
        np.clip(ratios, 1 - clip_range, 1 + clip_range) * advantage_pos
    )
    
    ax1.plot(ratios, loss_unclipped_pos, 'b--', label='Unclipped', linewidth=2)
    ax1.plot(ratios, loss_clipped_pos, 'r-', label='Clipped (PPO)', linewidth=2)
    ax1.axvline(1 - clip_range, color='orange', linestyle=':', alpha=0.5)
    ax1.axvline(1 + clip_range, color='orange', linestyle=':', alpha=0.5)
    ax1.set_xlabel('Probability Ratio r(θ)', fontsize=12)
    ax1.set_ylabel('Objective', fontsize=12)
    ax1.set_title('PPO Clipping: Positive Advantage (Good Action)', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    ax1.annotate('Clipped region\n(no gradient)', xy=(1.4, 1.2), xytext=(1.6, 1.5),
                arrowprops=dict(arrowstyle='->', color='red'), fontsize=10)
    
    # Negative advantage (bad action)
    advantage_neg = -1.0
    loss_unclipped_neg = ratios * advantage_neg
    loss_clipped_neg = np.minimum(
        loss_unclipped_neg,
        np.clip(ratios, 1 - clip_range, 1 + clip_range) * advantage_neg
    )
    
    ax2.plot(ratios, loss_unclipped_neg, 'b--', label='Unclipped', linewidth=2)
    ax2.plot(ratios, loss_clipped_neg, 'r-', label='Clipped (PPO)', linewidth=2)
    ax2.axvline(1 - clip_range, color='orange', linestyle=':', alpha=0.5)
    ax2.axvline(1 + clip_range, color='orange', linestyle=':', alpha=0.5)
    ax2.set_xlabel('Probability Ratio r(θ)', fontsize=12)
    ax2.set_ylabel('Objective', fontsize=12)
    ax2.set_title('PPO Clipping: Negative Advantage (Bad Action)', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.annotate('Clipped region\n(no gradient)', xy=(0.65, -0.8), xytext=(0.55, -1.2),
                arrowprops=dict(arrowstyle='->', color='red'), fontsize=10)
    
    plt.tight_layout()
    plt.show()

plot_ppo_clipping(clip_range=0.2)

print("\n💡 Key Insights:")
print("  • Good actions (A > 0): Increase probability, but cap at 1 + ε")
print("  • Bad actions (A < 0): Decrease probability, but cap at 1 - ε")
print("  • Clipped regions: Gradient = 0 (no further updates)")
print("  • Prevents catastrophic updates!")

## Part 4: Generalized Advantage Estimation (GAE)

**Advantage** tells us: "How much better is this action compared to average?"

$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$$

**GAE** provides a bias-variance trade-off:

$$A_t^{GAE} = \sum_{l=0}^{\infty} (\gamma \lambda)^l \delta_{t+l}$$

Where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ (TD error)

**Lambda (λ) parameter**:
- **λ = 0**: High bias, low variance (uses only 1-step TD)
- **λ = 1**: Low bias, high variance (uses full returns)
- **λ = 0.95**: Good balance (typical choice)

Let's see how it works!

In [ ]:
# Example: Compute advantages with GAE
print("Example: Computing GAE\n")

# Simple scenario: 3 prompts, got rewards
rewards = torch.tensor([2.0, -0.5, 1.5])
values = torch.tensor([1.0, 0.5, 1.0, 0.0])  # +1 for terminal (always 0)

print(f"Rewards:  {rewards.tolist()}")
print(f"Values:   {values[:-1].tolist()} (terminal: {values[-1].item()})\n")

# Compute GAE with different lambda values
gammas = [0.99]
lambdas = [0.0, 0.5, 0.95, 1.0]

print("Advantages with different λ values:\n")
for lam in lambdas:
    advantages = compute_gae(rewards, values, gamma=0.99, lam=lam)
    print(f"  λ = {lam:.2f}: {advantages.tolist()}")

print("\n💡 Observations:")
print("  • λ = 0.0: Only immediate TD error (high bias)")
print("  • λ = 1.0: Full returns (low bias, high variance)")
print("  • λ = 0.95: Good balance (typical choice)")
print("  • Higher λ → advantages depend more on future")

## Part 5: Test PPO Loss Computation

Let's manually compute PPO loss to understand it.

In [ ]:
# Manual PPO loss computation
print("Testing PPO Loss Computation\n")

# Simulate log probs from old and new policy
old_log_probs = torch.tensor([0.0, 0.0, 0.0, 0.0])  # Old policy
new_log_probs = torch.tensor([0.1, -0.15, 0.05, 0.3])  # New policy
advantages = torch.tensor([1.0, -0.5, 0.2, -1.2])  # Advantages

# Compute loss
loss, details = ppo_loss(
    log_probs=new_log_probs,
    old_log_probs=old_log_probs,
    advantages=advantages,
    clip_range=0.2,
    return_details=True,
)

print(f"Old Log Probs: {old_log_probs.tolist()}")
print(f"New Log Probs: {new_log_probs.tolist()}")
print(f"Advantages:    {advantages.tolist()}")
print(f"\nRatios (π_new / π_old): {torch.exp(new_log_probs - old_log_probs).tolist()}")
print(f"\nPPO Loss: {loss.item():.4f}")
print(f"Clip Fraction: {details['clip_fraction']:.2%}")
print(f"Approx KL: {details['approx_kl']:.6f}")
print(f"Ratio Mean: {details['ratio_mean']:.4f}")

print("\n💡 What This Means:")
print(f"  • Clip fraction = {details['clip_fraction']:.0%}: Good! (10-30% is healthy)")
print(f"  • Approx KL = {details['approx_kl']:.6f}: Policy changed slightly")
print(f"  • Loss is negative: We maximize the objective (minimize negative)")

## Part 6: The Two-Phase Training Loop

PPO alternates between two phases:

### Phase 1: Rollout (Data Collection)
1. Generate responses with actor
2. Score with reward model
3. Compute KL penalty vs reference
4. Estimate values with critic
5. Compute advantages (GAE)

### Phase 2: Update (Policy Optimization)
1. Sample mini-batches from rollout
2. Compute PPO loss
3. Compute value loss
4. Update actor and critic
5. Repeat for multiple epochs

Let's visualize the loop!

In [ ]:
# Visualize the training loop
from IPython.display import display, HTML

html = """
<div style="font-family: monospace; font-size: 12px; line-height: 1.6;">
<pre>
┌─────────────────────────────────────────────────────────────┐
│                   PPO TRAINING LOOP                         │
└─────────────────────────────────────────────────────────────┘

for iteration in range(num_rollouts):
    
    <span style="color: #00aa00; font-weight: bold;">╔═══════════════════════════════════════════════════╗</span>
    <span style="color: #00aa00; font-weight: bold;">║  PHASE 1: ROLLOUT (Data Collection)              ║</span>
    <span style="color: #00aa00; font-weight: bold;">╚═══════════════════════════════════════════════════╝</span>
    
    prompts → <span style="color: #0066cc;">[Actor]</span> → responses
                      ↓
    (prompt, response) → <span style="color: #cc6600;">[Reward Model]</span> → scores
                      ↓
    responses → <span style="color: #9900cc;">[Reference]</span> → KL penalty
                      ↓
    prompts → <span style="color: #00cccc;">[Critic]</span> → values
                      ↓
    rewards, values → <span style="color: #cc0066;">[GAE]</span> → advantages
                      ↓
              Store in RolloutBuffer
    
    <span style="color: #cc0000; font-weight: bold;">╔═══════════════════════════════════════════════════╗</span>
    <span style="color: #cc0000; font-weight: bold;">║  PHASE 2: UPDATE (Policy Optimization)           ║</span>
    <span style="color: #cc0000; font-weight: bold;">╚═══════════════════════════════════════════════════╝</span>
    
    for epoch in range(ppo_epochs):
        for mini_batch in rollout_buffer:
            
            # Compute losses
            policy_loss = PPO_loss(actor, mini_batch)
            value_loss = MSE_loss(critic, mini_batch)
            entropy_loss = -entropy(actor)
            
            # Total loss
            loss = policy_loss + c1*value_loss + c2*entropy_loss
            
            # Update
            loss.backward()
            optimizer.step()
    
    log_metrics()
</pre>
</div>
"""

display(HTML(html))

print("\n💡 Key Points:")
print("  • Rollout: Generate NEW data on-policy (not from fixed dataset)")
print("  • Update: Reuse rollout data for MULTIPLE epochs (sample efficiency)")
print("  • Four models work together: actor generates, others provide signals")
print("  • KL penalty keeps policy close to reference (prevents drift)")

## Part 7: Load Prompts and Setup Training

Let's prepare for a small training run.

In [ ]:
# Generate synthetic prompts
print("Generating synthetic prompts...\n")

examples = create_synthetic_preference_data(num_examples=20, seed=42)
prompts = [ex['prompt'] for ex in examples]

print(f"Generated {len(prompts)} prompts\n")
print("Example prompts:")
for i, prompt in enumerate(prompts[:5]):
    print(f"  [{i+1}] {prompt}")

print(f"\n... and {len(prompts) - 5} more")

## Part 8: Create PPO Trainer and Train

Now let's train with PPO!

**Settings for this demo:**
- 5 rollouts (quick test)
- Batch size 4
- 2 PPO epochs
- Should take ~5-10 minutes on CPU

In [ ]:
# Create PPO configuration
ppo_config = PPOConfig(
    # Rollout
    batch_size=4,
    max_prompt_length=128,
    max_response_length=64,
    
    # Generation
    temperature=0.7,
    top_p=0.9,
    
    # PPO
    ppo_epochs=2,
    mini_batch_size=2,
    clip_range=0.2,
    
    # GAE
    gamma=0.99,
    lam=0.95,
    
    # Loss coefficients
    vf_coef=0.5,
    ent_coef=0.01,
    kl_coef=0.05,
    
    # Optimization
    learning_rate=1e-6,
    max_grad_norm=1.0,
    
    # Training
    num_rollouts=5,  # Quick demo
    log_every=1,
    
    # Device
    device="cpu",
)

print("PPO Configuration:")
print(f"  Rollouts: {ppo_config.num_rollouts}")
print(f"  Batch size: {ppo_config.batch_size}")
print(f"  PPO epochs: {ppo_config.ppo_epochs}")
print(f"  Clip range (ε): {ppo_config.clip_range}")
print(f"  KL coef (β): {ppo_config.kl_coef}")
print(f"  Learning rate: {ppo_config.learning_rate:.2e}")

In [ ]:
# Create trainer
print("Creating PPO Trainer...\n")

trainer = PPOTrainer(
    actor=actor.model,
    critic=critic.model,
    reference=reference.model,
    reward_model=reward_model,
    tokenizer=actor.tokenizer,
    config=ppo_config,
)

print("✅ Trainer created!\n")
print("Starting training (this will take ~5-10 minutes)...")
print("Watch for:")
print("  • Reward should increase")
print("  • KL should stay low (< 0.01)")
print("  • Clip fraction should be 10-30%")
print("\n" + "="*60)

# Train
metrics = trainer.train(prompts)

print("\n" + "="*60)
print("\n✅ Training complete!")

## Part 9: Analyze Training Results

Let's visualize what happened during training.

In [ ]:
# Plot training metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Reward over time
if metrics['reward']:
    axes[0, 0].plot(metrics['rollout'], metrics['reward'], 'g-', linewidth=2, marker='o')
    axes[0, 0].set_title('Reward Over Time', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Rollout')
    axes[0, 0].set_ylabel('Mean Reward')
    axes[0, 0].grid(True, alpha=0.3)
    
    # Annotate improvement
    improvement = metrics['reward'][-1] - metrics['reward'][0]
    axes[0, 0].annotate(f'Δ = {improvement:+.3f}',
                       xy=(len(metrics['reward'])-1, metrics['reward'][-1]),
                       xytext=(len(metrics['reward'])-2, metrics['reward'][-1] + improvement/2),
                       arrowprops=dict(arrowstyle='->', color='green'))

# Loss over time
if metrics['loss']:
    axes[0, 1].plot(metrics['rollout'], metrics['loss'], 'b-', linewidth=2, marker='o')
    axes[0, 1].set_title('Total Loss Over Time', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Rollout')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].grid(True, alpha=0.3)

# KL divergence
if metrics['kl']:
    axes[1, 0].plot(metrics['rollout'], metrics['kl'], 'r-', linewidth=2, marker='o')
    axes[1, 0].axhline(y=0.01, color='orange', linestyle='--', label='Target (0.01)')
    axes[1, 0].set_title('KL Divergence', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Rollout')
    axes[1, 0].set_ylabel('KL(π || π_ref)')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].set_yscale('log')

# Clip fraction
if metrics['clip_fraction']:
    axes[1, 1].plot(metrics['rollout'], [cf * 100 for cf in metrics['clip_fraction']],
                   'purple', linewidth=2, marker='o')
    axes[1, 1].axhspan(10, 30, alpha=0.2, color='green', label='Healthy range')
    axes[1, 1].set_title('Clip Fraction', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Rollout')
    axes[1, 1].set_ylabel('Clip Fraction (%)')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Analysis:")
if metrics['reward']:
    improvement = metrics['reward'][-1] - metrics['reward'][0]
    print(f"  • Reward improved by {improvement:+.4f}")
if metrics['kl']:
    final_kl = metrics['kl'][-1]
    if final_kl < 0.01:
        print(f"  • KL = {final_kl:.6f} ✅ (healthy, < 0.01)")
    else:
        print(f"  • KL = {final_kl:.6f} ⚠️ (elevated)")
if metrics['clip_fraction']:
    final_cf = metrics['clip_fraction'][-1] * 100
    if 10 <= final_cf <= 30:
        print(f"  • Clip fraction = {final_cf:.1f}% ✅ (healthy range)")
    else:
        print(f"  • Clip fraction = {final_cf:.1f}%")

## Part 10: PPO vs DPO - When to Use Each?

Let's compare the two approaches:

| Aspect | PPO | DPO |
|--------|-----|-----|
| **Complexity** | High (4 models, RL) | Low (2 models, supervised) |
| **Training Time** | 10-100x longer | Fast |
| **Stability** | Can be unstable | Very stable |
| **Data** | On-policy (generates new) | Off-policy (fixed dataset) |
| **Reward Model** | Required (separate stage) | Implicit from preferences |
| **Hyperparameters** | Many (10+) | Few (2-3) |
| **Sample Efficiency** | Lower | Higher |
| **Final Performance** | Often slightly better | Competitive |
| **Use Case** | Multi-turn, long-horizon | Single-turn, preferences |

### When to Use PPO:
- ✅ You have a trained reward model
- ✅ Task has long-horizon dependencies
- ✅ Multi-turn conversations
- ✅ You need online learning (model improves during training)
- ✅ You can afford compute (10-100x more than DPO)

### When to Use DPO:
- ✅ You have preference data (no need for reward model)
- ✅ Single-turn tasks
- ✅ Limited compute budget
- ✅ Want quick iteration
- ✅ Good-enough performance is acceptable

**Key Insight**: DPO is often preferred in practice due to simplicity. Use PPO when you specifically need RL dynamics!

## Part 11: Common Issues and Solutions

### Issue 1: KL Divergence Explodes

**Symptoms**: KL > 10, model generates gibberish

**Solutions**:
- Lower learning rate (try 5e-7)
- Increase KL coefficient (try 0.1)
- Decrease clip range (try 0.1)
- Use adaptive KL

### Issue 2: Reward Hacking

**Symptoms**: High reward but poor quality responses

**Solutions**:
- Increase KL coefficient
- Add length penalty
- Improve reward model
- Clip rewards

### Issue 3: Mode Collapse

**Symptoms**: Model always generates same response

**Solutions**:
- Increase entropy coefficient
- Reduce PPO epochs
- Decrease KL coefficient
- Add temperature sampling

### Issue 4: Training Too Slow

**Symptoms**: Hours per rollout

**Solutions**:
- Use gradient accumulation
- Enable KV caching
- Use mixed precision (fp16/bf16)
- Offload reference and reward models to CPU
- Use smaller models for critic

## Part 12: Hyperparameter Tuning Guide

### Critical Hyperparameters:

**KL Coefficient (β)**
- **Default**: 0.05
- **Higher (0.1)**: Stay closer to reference, more stable
- **Lower (0.01)**: Allow more drift, higher reward possible

**Clip Range (ε)**
- **Default**: 0.2
- **Higher (0.3)**: Larger policy updates
- **Lower (0.1)**: Smaller, safer updates

**GAE Lambda (λ)**
- **Default**: 0.95
- **Higher (0.98)**: Lower bias, higher variance
- **Lower (0.90)**: Higher bias, lower variance

**Learning Rate**
- **Default**: 1e-6
- **Higher (5e-6)**: Faster learning, risk instability
- **Lower (5e-7)**: Slower, more stable

### Tuning Strategy:

1. **Start with defaults**
2. **If KL explodes**: ↓ LR, ↑ kl_coef, ↓ clip_range
3. **If reward flat**: ↑ LR, ↓ kl_coef, ↑ clip_range
4. **If unstable**: ↑ batch_size, ↓ lam, ↑ grad_norm

## Part 13: Generate Samples with Trained Model

Let's see how the model improved!

In [ ]:
# Generate samples before and after training
test_prompts = [
    "What is 5 + 3?",
    "Explain gravity.",
    "How do you make tea?",
]

print("Generating responses with trained actor...\n")

for prompt in test_prompts:
    # Tokenize
    inputs = actor.tokenizer(prompt, return_tensors='pt')
    
    # Generate
    outputs = actor.model.generate(
        inputs['input_ids'],
        max_new_tokens=50,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    
    response = actor.tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"Prompt: {prompt}")
    print(f"Response: {response}")
    print("-" * 60)

print("\n💡 Note:")
print("  • Responses should be more helpful and coherent")
print("  • Model learned from reward signal during training")
print("  • For better results: train longer with more data")

## Summary

In this notebook, we:

1. ✅ Loaded four models: Actor, Critic, Reference, Reward Model
2. ✅ Understood PPO clipped objective (prevents catastrophic updates)
3. ✅ Explored GAE for advantage estimation
4. ✅ Visualized the two-phase training loop
5. ✅ Trained a model with PPO
6. ✅ Analyzed training dynamics
7. ✅ Compared PPO vs DPO
8. ✅ Learned troubleshooting and tuning

## Key Takeaways

- **PPO is complex but powerful**: Four models, two phases, many hyperparameters
- **Clipping prevents disasters**: Bounds policy updates for stability
- **GAE balances bias-variance**: Lambda parameter controls trade-off
- **KL penalty is critical**: Keeps policy from drifting too far
- **Monitor metrics closely**: Reward, KL, clip fraction tell the story
- **PPO vs DPO**: Use PPO for RL dynamics, DPO for simplicity

## Next Steps

- Train on real data (Anthropic HH-RLHF prompts)
- Use pre-trained reward model from Phase 3
- Tune hyperparameters for your task
- Compare with DPO baseline
- Move to Phase 6: Multimodal RLHF!

---

# Appendix: Quick Reference

## PPO Loss Formula
```
L^CLIP = E[min(r*A, clip(r, 1-ε, 1+ε)*A)]
where r = π_new / π_old
```

## GAE Formula
```
A_t = Σ(γλ)^l δ_{t+l}
where δ_t = r_t + γV_{t+1} - V_t
```

## RLHF Reward
```
R(x,y) = R_RM(x,y) - β * KL(π || π_ref)
```

## Training Commands
```bash
# Quick test
python scripts/train/train_ppo.py \
    experiment=ppo_gpt2_synthetic \
    device=cpu

# Full training
python scripts/train/train_ppo.py \
    experiment=ppo_gpt2_full \
    device=cuda
```

Good luck with your RLHF training! 🚀